# Which plants could succeed on Mars?
Lissandra Kruse Fuganti · Cultivation-success evidence pipeline · October 15, 2026

## Research question and estimand
Compare food-crop candidates using measured germination, day-50 survival and flower/bud outcomes. A successful pot meets a stated minimum count out of five seeds/plants. This is not successful harvest, food safety, or mission-level self-sufficiency.

The estimates concern an Earth greenhouse experiment on a Mars regolith simulant. **Actual Mars success probabilities are not identified.** An optional projection below is explicitly conditional on an assumed transfer factor.

Run all cells using Python with numpy, pandas and scipy. All input files are embedded, including the real experimental data, the original supplement, the ancillary Kaggle dataset and APXS chemistry. No additional Input or internet required.


In [1]:
from pathlib import Path
import json, gzip, base64, hashlib, math
import pandas as pd
from scipy.stats import beta
from scipy.special import digamma
OUT=Path('mars_cultivation_success_outputs'); OUT.mkdir(exist_ok=True)
SNAPSHOTS = {'wamelink2014_pots.csv': {'sha256': '3aa6c58f528c8a1b1da2c3630d0c4ffe39553383a67d0beb1b36f689cd1382ef', 'gzip_b64': 'H4sIAAAAAAACA5V9y5IdOZLdnl9RH3AtDe/HckbSzGZamzZpz2azRzSxyDZWsST9vXDOcdx8XACR1WmZVdaFiHA4HA5/HHf869fvn/737b///PVvn3/c/vrPz5++fP7t9tfvX77evv375x+/fvn28ffPf799+4/PH/8Y/+Hbv339/n8+/xj/8i9fv/zx+fb7998/fv3XL99//fjbb7ePf/v+x+d///H957e/3/72eQzUv3/wt3D7y9Mv3//xjy+fxvu+fvnt9peP4x1p/Ljx631+yrcUn/qt+qeC8en216dfvv6/T9//OT725dPPX29/+f7923iP409+ivx1T36MjuX2P59++e3j71/++Gj/0HBnP7cb3tlu//Xpl08ffwySb//t44/f/9f49HjF+I3hqd58fBqk9Kc4xvqM7//4/I+vn//v/Hbk18MtjO+G+7fb7b88/fLPj19//vb7j8+vp9X8U7tlP4ZG95TH2ORu/2F0jpeKhGyDU38aJMZBR8hPCYMx9p+fv/3957dPP79+/P3nb6+eGH/TYJpPeKI/tfFEf02K+CWOpUGJH+wNTx2cCA/cfc2PVJ7CLebxx/snB36U2788/fLr92+/f/w2mReMH76OGYL69hTw8v6CzeRHHj8guJRBcART4qAabw23//H0y9+/fP/y6eNLesd/GUsxeDZ+QW9Mj+v7moYUBq0hDSJ847LkR+69WpvSngr4NmbbJXEVPPn0+cfnj1+NeRoL6YBkBBfBRK/hMWL4xx9/fP72G8T5Qdp8fckxfvvVf0/hpSzweS3smE4ZKzW4Wm4SxhAfN4P4atunBkjuoC9jXuCYf7vfxK7I9SWnCpY2UOgx3r1YCY3FS7AUbowaom7i7turaYFsb9MKEMchCp5iUBYLIKZGjo56Jf4JdvpX7BQFr3evxws/fvvPId1f/vH965efK64HtxgFXmkbjJlDsoLEi/Ppjw9M0dL8h5BULMkgnKsR++3fhmr4+bcfH6de0KYRUx3WLlK8Y30eOfeXJGq8c/K/if/+5U54KagdOmSMTuRqfbXFX750/G1QTNi5IWhu8ZUSe61vMBgKJFJ1+PRa373Szq4MDnQHFVmoDWJ4LfwYPcXFh0Fwh86jyosPiv9ZlbpbHftvKPybNF7MCzX+sHGie6lfXu4bLNTzOiX/aoNNRRTGr/eQ06GOGpUo5Ca/3P3PnAr4rVgDX6lDXRYH2gsheKnjQqhjAVrBPkjSXqG80ixz54KQ7CAIcQjBePXgcPsQbjW8nd9kLbZM4G7sY1wquwNiLlzGukUoxSEQZTxS6uMs5+DQPbRn5pLgIZCSj0puzMDtT79C5YqdFiu/Xutmee+ik7B3IGcQ9jHDvBSeFxuzj5ERByZnFzcaQgoKBgAGNzu6xwNpcwzOU2WMHeKf8SeOZR/rGlYHuIRqcC6DFGyPIc7j/6qPNsRkznjdLVAJgfJcHnffFKgKtVpAQyXRqW5OtvmAjybXY88GPZKO1lfBOR/B+2GmRJCTF3ojvdIbmAGEsLZry2uM8juToEPy8zAchm1EVqS2OTf0RITuHo9BcwYSm/pCJd7NCBzZ2ItjbbDg/eGMj5SOeGsVhBQxATMr7uGgezWnFvfqBQIJS3ZsQUly8btT5i77Deo1UfYhaSXvzvwspTy0/bD4hja65chVzumtMRXtqA3Y2OPNTmIZ3x41U9zxIlh9EcYHVFH2jxZl5ksTlwCCJlmvcXF++2kacMfNXVfT/vAYxsuNB9g4PweL3UbJ+iGyQ7ph74ztAQbnfrK2BqVl7U9MdmacHYlHXNdOT1vzLOIgmIc9Pl7LlXnqHczNDGWFP3jGbZ2RBF55OEOwOjC59mh8v95f/a2hMec1zEEqYfhIEuuwNZCMVPgqtwblnbx2TVucXxCD8bdEOgodqs9FrlqKa9vzlW8R4AoMVuOBHB5sz6ni4bJwUcjn5t9YXndzKsl9i7Cp0ofxj2Eyr4w0aoQCR4UmB8a3y7WLtK4DVQ4WfDzl3HFFhjppV4dzxNYd9hS3fscjZamlp9c5pA4HMxk9lmV4XHHl7s0dB+PaZf7Bu+vGDKM830AKHoFtA570q4Ma6zKOI7iVfCCfrfdBrNuo9WcdRbsGDgQZ3Nb7NdoEYdDcaMV5e2IYu5szptKAh4hkTbCHg88OUaY2oFMNUXLXRxy+n9ZmGzjb7NdzYH00qqYL9XbdfEhrg4PmDHS6DzoO+eaw0CmJ0kMLFobpOGTB3r4yd8HcIZgedpXvNHtCoqwNs3drW5l11SXOw5DUJ3x5VPOvt0jfGDJ3fwpv87S6vZFdLgy2FOSkNBhi2FS97S2r7GB7w5YZT4iBbqPphtaKsDtCRYAGRrJEI+wOqN5kAcAYqFp3lzaaN9mJmmLDJxocIg9TBPSXx0PdXFmqRs+IDkj3i/AFFjRSMSOQA73ecbKO1/qF0TTPDFhi0kz31a9rY+TuOwSaGJ6bcrg0VE/1YEQGeCZ0eqrID37tKtONgFeF+GCVenJxZ0xNyWnwQSMslBQpOT2/tXnmSkXsTF/ktIuU8nDypxmYqVx9DPaUgN73Jr53tDvTXCHnTyYKBoSTxQnK8tLkjDdJWR3WOIzzzpOtp4dTdU4jRVO2KVB3trw7sc2PxKnj6UyOrfUB+yW9Efz7wnrIZMP54APigRic35zFLxSz0wYc3krB0Jj3IczMYAVsAZFQ9/Z3hLg0HlA45jHaXR3zsuoctFjSF1LbRkllduQsuyrjCCf9oS3P8OmO0L/FnIdEYLbtcXObNuU5kOkeYmBf7KUp7B5aptIrE7vL3ketRaueyXZOMu89E8ketmmz4wsP9JNjMliRYcT75LW7+UwsCwvPrCv6JbD0ONPc1s4BfhKM/kjjeK6qvw4YgeS0jVm+PnDxynB92oNOt7ZPko3yFmGUQYoH4jK8zPhSpNSFyOhHEcuyX53QWsU0NDQyFQw4pCBhyv0YqwEb6jE+4iDFWX9srWN9qzZfnM3VrNcpo+Fo4XHgIJ3BPhFcz0YjKA5HMxsvSQ8mjxxUmwhjrGNzcGnbVrFHnl+BUTgNjv4igZCqrIyqaCuodfsYasXkc9TBqs0fF0fwjM9HWrawQr1tjBJ3B2SuCvAlZVXAlLAMOCZGk6D7Q4dplYJp85zfms5p5gloCvJYNybmsou+wuKJjMiRiOEGHo47MCA9WuHT2aELHLK2uuPr/CohZ8L4IvGAoeXC0IoMaHsqQ08zH+S4s8cWb6UyQAo/wjZ16Eu3an6nQ9cnHMwpinuhXuRlaKcj48I0E4Q4HbcAvEF/3PgYUTan9dB2g0LatG2YDvjeMNt22TvkPBNOpgBG4719eQRnRtgxk4owZ4KUYSZhtT3EqQzDJWPR21huDC4LlT2j4IEr7oacjZE1XrgRTDjSM+GxgTnmB108hdh3z2gvkm85ivDqN6sssYARpfBAhNoE8e7tuf5qPXzzy7zMjG09BWZTq15W8zLlY3OzAHVThB9z8xemegyMI8CI7UWf6GF9mE3fLtO4gAWQuex1GVtBlGtQQ81ii5hPKgBzS5szY8YxSCu/HEhoX0YZneUp77+kMuwsvjebw+0tjgppzwy8VHtrvAiNBMv/wSSjsFW3NAnvATomgIKyJJqkOzq5yFU9MenumHAHn+NJN8K2kiUJ+tO1eYM3tm2U0BbG8SwLFNREuRg+79ujIc54kaMbNBwU7tng2hoHgbQ98RKhyemT4qhX/h7z3wlnFRxFcr0+OhL3dD5d/iqfpnN02On+u5GdGDxByJuiXdsiXz7lMJsciivp4TBOnOr4W+hkFwIzXDKlmncHfYX7wPN78Ei6r+9C+YHhykSWiyHl8bS9J9IY6qGajHbq9I2peE+jFyXFq2033+PaXlWoE+fAeCltxqiDpvTdUaiYpGN8IzNpgfeXB0PvvjZMnTjGdblJvVsYephswIpn6jMYhh/KMJ/POI4xIq/zyIled5ejGeRKj9F+AcKZIRt6irSQCobGHVbhHuSE28Uc5DgZ+8v/NZLettEnJBoaA8yui65whKPgbXWdQ8hmjMLXZZwMk9zAS+4ChRBW0VP6ejuFNjDgAQn0DD95Th6BaWVr27ZIKAFt2wDwEqfVd4dVfzLFNMSHI9+BPoA4uF1ENuO8QB5OEUWuUVjgyO7BSoIkdKzixXtAUIaEBcaSut6b3oF4w7AtcOpuTXVzepRWAIfj2cEPTuAVKCLwms/0Rw/CwnfgMJ32aCsY+jniMDVo0B8+Uo5GMCRyDYzxlsGR18njB69z23BEYiYEjozmlU5pQ4czmxA8B71MUtMDOmX6mHQvASTrenfwO+c6uSdD1QU495jeBRgwyggshIxR7kPc2piJadZkMAvMMmwtDglJVVoa4Te+O+2D6p5YuR616o7El2NKn7kD0M4UnjZ5XeTpgyn7ShcJihfhTn7ggMGszLkGuchSXmtIUp5JcGdwDyyuhOAdyDMMa5vsS5LjzyhPiNL9KRziXF15iaZIJEb7g90LpTcjaCAjrzE+EZuR2ZSAiY3Z2fR8PgJKhP8iNxwxPXwmbOKcM3zpmLYFnGSYCXUJOwq0gTyzDRXmVYEyQd5tl/YfNj2MXh3fQ144+IDISZwt+AOrCaPbFojiaRAmCErNoqM+okXueelOGzxhSXPWu/MeiELYYTfH3ovwFs/hL7yxXbhHRGwpd4vxJa3htNi4xcCWmUx7m0ifrgV0TqH2bSRgDcB6TWW9gl29Vud47TVyDdYfYQvQDZxbPvnMmQhkWHYc3NfYHAKOsLkY9iAqDQtxid5oXcd6x66kRDe33evcJ74J2inS3SrQZaq+SilU/OGrczqkEIKlG7o8GK566zvDLfcnyxJTlWHsAlv4DN5VOpFLVMo+lskYUET+sUnyaznFD7xcC8NdcIp1G1uNVg+QhFwcRp2zb2zwUHcnoAJz1aDtu7dFWgCNgv14BoQizK8WJAflARd19+mTskKFRzeI8VegYc9nCHWpVWJQ1zAaQgcJGo0+EJucNL7lFU7A38HTBqDmq3dAxnsUoAo3Aqu8c4es4E2WI6KL4gTV53Rz3LlqOpNjlwNGXDOfKH6Vx1ulXDD4Ebs148CCIjBxbQK8w5De9yrho1l7VXw/AvnwzndgGEHmI0zpOcgSLciiL+4wVm9eWPdhG4WzCb7n0DMCbOyu6NdagMcqnYeAdXJjbOibfG+mdGVF/eVFDW2AB/oV5jQjlZuZHnfYfg1B9h0e8bUEgPh9YrvR7IDkEgyH9x7QO5mahmftmANfvQDQpCldSFkQNpZFcrgEg/HEV7wEXjw+lo+hEgf+d2HNx7zdy/+JTXmH/JfVCi+eEBLQt8c4EUJb5WZx4s5vSwUYX+sGmMuaRriAcEEKyiaSec9DJUUrguQs+rJVS/dagU4XuGt+sSxdYIQFU1OmjdgRMs4vMHhTY3cGogNyBeSFu/JAnrTlnOjw4Z0JY4w9ItVo7w37lgUcZEp8B5wM67HH4Ll5qL7IMIOQtA0nmZZS1Qkz8/Ci+JlyiH474QQcjQ6O9sskNmOZQdkxRnorjHOQ1NeGPwMPnmAO1wm7iCa0cRU9mwHBl2YkBm/BNyw+sdA3eRPjrmYnMOeUiRTnFNNjEPuemyOopwntAQIO4DJUY1XktRJOIwls3FsNppBYAVGyUDyFm64eayBYs+gIMxMHfb2I3ihpT6AL8z+YRz3Z1pEwJhQhSVeHZTIRNgaDacRndNtIoR6jqxgR9pa0UERKhlKeglvE0mErwJDG8sCEAgyx2lGRtoGeyN1eiVYE247gLKxD20dgLExahYAsMLTSh6HY6juwJB3o6VN5H0QJx19u5p/ykXSyaWD79MOSprtVFbsozflKNCMtcuGvHL+wzZff81SEHT45RV7GI2WBaYkTpCgPWdGjMVWMj7v0urfiMTh8nhUIePsCOhS5UYgcBfdgUaGUhovziB+buswzR1oAZihNDE95lU9ySqmyCkIAKdBRHiOAcZauKPYG0C5pWCFSokEqVQ/Cghu+9ozvwdt2oDwfo6CQDlV8XIp8RGdgwmFbvzlTs8kcRnF/gyGJciVgZsCP6vz4Dlh3hxpkFaJ0KF2JjjvVg3srGmF8lcNXsCDtJgIlm+A1kv1yLBl9e+TjiXph8rH4in9MJmrc+ehJqTFxR/T4/bGmeghAuSWV7QA0wH/f4e5eZin50bRHI0WrR3ZKADXy9wK7hSm/A5uHD/tr+EaX800IEhi0xVnFGzONSSYlBSdtIWzQCQRDDqc42R6LC2BIoFc8DnsCGokjiSb0sW1KNuZ+h90njEY1eqq7ysio2Jd1sZxtWJm5zvCkkZXyQfuqbpCfsJ+ZDaT3krIUeAn7eAShOFH1kkMTcXzNu0xaEyAaZgJJTungobF2ECAqpG3JxHpGH8co/UE4af3gUbHvD9hyj5KnIuxfxvDk8iKdYD+Z3RxCpgkF6eb7F1CluagFijkyYcbSeZRJ5d3ubl7uEGENXcTEPbAzJwICsp5Kmm0KbhmrwaoW2pfEZGFzgPZ2QisRJQGDF+XRKCgHQWEHZY0TLsGjq+nYIn+OwB5SkZcFBlLBiJiD4MI/Gr9FMsQiXc2iRCeKy0K7W8izqXKnygklseWdzhwHp1WfC29nTBN0wLjQ2iYgNCseg3oMQELeAbvhK92hUjdquVHZQgy33hwvAmKFxyL9oS4RSX4LjXrGlEOFZqJ1+5QTf5UXJkSAfmmPc6e6dV5fZf8GcxdRZxgjXlb7DnZg7hPMtszYldju0hIswMpF5bw7sTdIrYv/ZR+q71TMNMXzFMQtWBJtbLBIxG80TdDtw9GFKWSWQ+Qptv3KyODeb7OTBR7ZobbWmRgS1ffgllRVMJTvE469XtUxF7gptJ8ZNOA32vpYSpYudph+x9bVNNoeFRheoALpv+mBKwhoInuT2vsULV1b45gz03+I3Jf70sVWV2GJaODIZODI8iq4Zxu/7HKZ7HNQ6YmhBklEuYv6tkhEEOSkT6mqm8gGY2tVG0JrF6/aEQhnyQmHK283gYjgFPIzlvaLHi7j0ImP0Is3I/J7ejmhfc4eVTDx6JkGEhun4MVrJJAzyCZTy2AV3t1X3t7UW92KVuy1dVu5ZsWIlKaoF6dLvjYzjQSfwEMpr91mmpjqXAXgxGxNhM/s2m0ZXBIwJVRG4SxpemIJoYjzBGaOUvFakdQPiq8LFsMKIwwubpeiT9hs3WSp2LvjuXESRizaRs0DvRvwQIsTl22rnFUeeTuAvMZuW/DQHQ0CTUDlgAF7sIfK97viQBqd/fXOyG4dNgjK9jZFek2MLto7MRRPrAfcGxG8BxuaaqF+DIE+hZtMCStc1RvKU1k6EvjpuRDYz/JN4unJ6SMakCPiycAPPOKbjnmw7qIXF4ZcAuGYHWXeOU9BPPaAc5bxy3p/OFY7k5U73FGcDUAKAVzQROLrezpeYVxdeK0Wk2RJMdAq6K4TbUOGfHVeqhBZoEDtpJROXsI4wlnXGPSceJKugITqq5TleyWT7bxFtTIkRv+xNBtc/DaIx3ZGomoKc84XtjK6tcBWJmpCz5SwSUlFQRsT9WI1euIBGVeCTMKg+DCHb6GQ0xiJkRZCY+WozDBwNmwDaHGGzpnvt+yQR5ucd2S5Ma7WPWJV5x5DUHrnuT8UX3dszMKXhEdLLFg/L2XOuCGAzDtgvZT3J2yoG3WtLbMFAMIlIWqbKtuQyeX79z1fWOhHtDAhzaR72cfmsRaFpFwBNFKWYReJUBfnFmiuaYI0Q/zZROMFWr9k9ZDKd4JKP8T71IfAqTWJSU+51J9ZzOkAYomdhw5JQhOy4WecPOrLDgaskGU7hcAeHhWpQpLkd5jjTNMGgZ04X97dJiCcCDuEjkg0dMSdcmWmZcZcmEJxkp5Utn1vrPg7z55/soxA1WVHNIKa2PsrSCj2IBQBAc0rwssvAIYc03fGqxygIGBa13r6TR0/W3KhUPM2ZbcdMTecSNmH2GpQvms2UgRz60U5LuIOXpHfUm2HtryrhZvxArZVYxi4TcrrytgOzDGGHCyBXvg9CUu77O3F3FmwMkGtfHWrchw7hYoKLd2zrqln/A+HLHt1WTbcOkplBfDJ0HxszMiYotDCZXKz7FIw3rR1JIg72H5o4VhDFlCnDG0B2sp8qJ5gnu1JXSDC1JBxa7FHeSEk30t6t83EOihXe0vX5mZu22r2mf4qDKeEqVDTIRVg7fXkaAIH4mEUt30C1VuOulh4P/CJGLY1ULNQW4VtHO3roQKA5YMQYRa0YXgoF2WHposCCxvxREqHZohR9bBJVXyk5zEZfwflPsmirjZRfwBW4LRGwEgNBcTJE0aCVQvWLUeEuHL0jj3VOk3gGiZz2q4O5N4YpikPwhACPrLB2L3ctZjpoYGMGmxlbSiMDfvgJKG+BMyGZMPDEnwzK+AJfSvseJ2ysT08giDurTNNFP0cWq/guPyAxwFSsvEk1hOiNVsdvzie4rLXW7rJQY1W8idi8jscJTAw73KBmTaAV9MBL2K3WDu2SI1BTQBtZ66BKUq4eLambgxBxvlA2pUJQYgyM9lGh98Xq0fkGIhOT5Nv4dhEiC+8QPpxTLsy94Rw9/TCpQ68P9Q/xGrho2eeuX480qPpb6IHeKKDrrKPoSt31mS1mhDtu0plrXUXBIaci1d1aUzn8o+zxfFxj3OOVrs+t4x3F0eKKqzCXU+lsMNzVPJdkZq7ui+PmZZZIOTNXRDRaYW2VSsa4Pfajd0129wI4dzrkEPSMfHwkGjDSvZr09fqM5kHBd3vQE5yXFmW/z53qYtsdTh0Elokn5s8EYVb4bY7dibWM1eNdBgsDROgj0dqWhzdz8V1TLJATtAKo+wvCYhNsQlrSEBijlAwjMgnLBhb1XZBO7K+v4MRsZ1JZE6Qex7TKo9oyVnr7Hh4+fnWvNHngvhHFQxraNu44eHenLYYFgCzc+fa/aBiPp6Ptub5qrePcG5FkZ2imR6wazGpDJSdtl003rQt5qdmK4e9pbnopZ/bdqAvSTUnpzmjqe1aoLGdDYIn8DlyM76WczclvvGyTVphGZkTrMFWIC5z/YaaVRtJbTsxJu+hZclkRzsiTeaUP6de8MihFxPXiee3AuoApfSjBZkMpu7nbnbn7qTRmprFyaG8b9WaGOBiDYnevWveNAxC5lWHM0iQyJTmemiZpjZdWX2knKZaT8U8zPYU9nIKzZ4o+VzthPbWMkEYEhGLclsfRzNGwcJ+vT4urM574Rh7zD6rhneAysiTvrM0/WtAPkldQMSs2sMrBG4tj/jmC+QbqXwH7pcfTofCQDa7C0Km2zbw+8is8hHuNlc5l62zlQUYkxcNgWjnu3nQm3GWeKUpc3FToO3Z/6+xnLaZVYrx+dGYvrf3jGp7FRk3zEOYyraNrVcszFlYGyWOdd9vgz1BWPg0B8eLwkO2pbYueI6PtL6rlbo3Sw3K1wZv9If3gH1Ajb9UOqwAgjmI4e1wRcuLLQUKdg2/QrBiJMSZeYxgdD4UkBbB15RDGYPdsXdVn4WSSRwvoe2MmWytIZj7va9muKoMJvhBaEDuC7wmHHS9k82SWb7AT6TLyE61CHvXA3som50l3jwPJ/ENp1gBFWyz3s3ZmOrdNj0QJgjWasi7LbG/QmMF5YCh6GqfIrSFNzFYQ+ABT0XOvMQV8DDMHvj93gPfKFo3OmJvlt6YWKMmJaSesnHAHhX2WpILEKfo7RFRZkljMYi8CnMrxnecF3j1ucseh5wxj/jYAeCVrTQv3wQyBb+u4DJYsL6OWASWWQZdN1LLZFA8FZQ2u1RLG9PHdZsIdQBn9LfRChY2DrSssYkbGwzklKNnAo7VDSSH4LzOsl9j1qK3mLvbWmZv6atu5wcL/ZbvI0O4rPZsAjkRrSpCXPtztigeOfQvs6YmERqkMb8dp+a8wEOCKXlznQkRdYTtdXZ85+C6K9BoXi0tJ+RoeGLFb6/z8PKnMuFSGp0uO/rwDOEfMn88U92qLlfyp7Z7TtGfyOG5LqNgrBxSdzGvuCBA5XwgXPj1aMlp/TZ5UxWe2Xb7ucMIq5DQ7MvDqZ9a4ST2SMCuK94+EeshSKY9Wu88SmEHnnXmYLg5231XrFJV/EMpRrWJHnDbHqlhduBo+nOnpm6jww/k9EOdsrWkUAwBLNldERZnJ2VPXEyaLCwX8FAuLOP9+kC5vD0w3nv5+mCzjScIXbKievZQz1NENzA9/shsYac4FuCKsB0e5R6WJriUFiYDBXjmPfBI0H+F8amMO4Gg7IyzaQF4um96nh2EuYqOBVxtdguttF3dnGXcR/2ZOGbIUzwsfXNtiyLiQF5q3+ULbBGSGpmHJNsAmZ6KhzbqVf66fAUwsO86SLrZhv5OTdn2zstp3uoZpzim9t56BLy5bWPAUQ1MdM+cKVW/iexKGTGr5W89zSlu2/g5q6JvvOorT8rP4F2M8NsyjjY7MxRjWmz7jpSFEdr6vIUP7bJmwx1FdyUdbRFtNUc5queDutWKa2F/4QMdO2ZZ5hnh/1S3IzLl3L2OvEjnYloaIIpAw9sYm6Xeqjtgctxzy3V/t3nq8Lb78hyNtPqixS+btY3kR64gKq3J+WGlSNYze7ANzQwqQ2BLSFLzKw/ekBNBLbeDWrVy+AkhoJLjpAiWaNkiXNQAUjcZBTEnX9yzWhjnZaebfH8oHYtZExQt0daBZREkysd97TEv/qrWAs7WIK0qmxV7TtZajtl/MbT7P9dgkRT5TXg+eV21F6ygCiuQDlWnfFnYOf3ElrEdUbSIJKZ3BVQa36z5YCR4xW0IDfGTyLCNlgTLhwXtieq2TeEqu3rz4pMs2S5te1VjUmJYB4+fWyHsjslCs8Y978wF8ChY19rgKkssYQp1+o+gZHuPX5g3zwivjxXZgl/o4rBzQ63aMsNgPAPZOo+ErHQKKL8C+lRwPGerhtAzZ4AYqKhXhcEM1NR5HQjf2pYhiMi6U956gjtp2GIrioeHFkzF0rFh8tvt7yuIWYjJaIYaaCnnskqmFJmlUSwTU27HO5ODarALTSHRX8tV42rea8hz3NTnGY8LWUn7azKyLkjsjMBJwC8gwBjyHuQxSLsAuYGpV70GyVX+mfuq9m34LiiQ2Ox+UI5OB7M/CNZEZYPMygffbjX7bd/kYg3J2ceBg4PbrrB7gVUKSpvigba/NKFmXS+f1DoAw+NFrh70prVyDnK8KcaVnp+GvyfpDkKP1301uzvaDimM99uGvQ92MIYfsE8lqLdHs4IbDC+nIwqMaidX3yrbvHyCyCdSvyge8A4pxRyFJu36TF8XezNu4YNljQlZFeG+HDsKVF7nGWcVPz9xcTebz7pXKMrT02fSuS8PWRhX/o/55JFapd66EZHLskTdkKQmU2HKVLpusxYVZEmWGsBDB+hPz+q30Z/lNl2hnMDsetFfJtzU4V4y6NuVsg12pyfcBZHhrzrnsQKUV5QkxsXBzLjEWTtr6p8nUBKz3IHDPFvyZQol8RqSx7TvtpSyEjHWiZSidcaS4X17kFwTqp6XaIvYvL8tkdcB6iwsxoWwu78uveqCQirS7k4i4ko6MURGQz7c2sx6E5zk8NrFgbKNIoZ586xXm55uWyvmC3CQ+oH02QGDc827MD80Cdv/Me6owf0qSlNvc5/7fNGon7DO/MxJtzWgkmNVCO1F5A61J+IWNPdW4fpD9PCVN4LB6eKS4tgZsVPxVSZAbDhNtV2VT88LagJDQuORFrZJq6KuSURKB958wAfWmW9a6o0ryyyXQDcYnw8w0qxoMswqDb7AlfgmFGVRNhN+4nugBmNc2bfZCbNxDuMzzgDbYKfbx7Z5haRa83ibad3dsuffnudjcG5HuHzLOkoTr8YRb9qhm6+rMvs6keHiTNppm2qNvfDcnfNp21KkqhcO3Ow6Ba2+A3CMcce7EpEqhxeInlzM1Wil4ilDIPx9t6wrpunekcSqujOI7Vz5jbounGAZRLnZPRq2RcoVXItRqTwRjGDm5u5C9N1IxG+HomIUZ6LWy75FBFNZKSnebpNe926L7P7DAlGefT4ZT9u5dxpfuUDrpXn9SbXLBKNNsBzwXK/QtpCBctWWs3FBO4JirRnFdYH1ue895jecEGMYewBQZSq8pnAYprnCVkZhxbyB9NjGidN0V2Vw7G2LHm3Pe6P7bVJDmqZG3Zra2NpHH2r721+qoVukBUo/uRlOnmnnReZkzQXkEbKxuNAwPndYrYT+2bbZ9rViAQa6STHJNjXQDvKa3bTi1dIIVFzByvD5PaaLsI3pxmLe+aomDUhvz0J/B3pEce7HqzEY5fAWA9DGzRs04TqjMdzf5v3pLtukKDfB8Bp+efmXV7grqdNq11P5ClMI3LW+EOsm51TZGJrmpl4ad3sysRm7MkKRY50/ltfpyiroNGQ1SYXbda9kT52o9CTbOZCWtO8K99qe4uh2vGnKY0/yoEMAIr26aUrP9z+V9OB8tteYJS+3Tk62uNUOrdWz2fpV730HuI6vzMsC54nPTVZpEfTSbZcetKyE1910VosbR1wLP355nxxLmX2a9+fytWFvbjWaNk7Wma2J2wFsn69UFstC2FmVFuvwUpsVXTTxRO39FAEOvDcUN7yjcjPbNnJ9V1zFK6aC1Sbagu9BePOyb89lT3M/1f3F4N5u1XbGGeeOMRRYCB7EO14RLhbVXS15R5qcl4IWIyUebj1lIQCurFARMl+9BVoBEMaO3FFtCO4K6eqOyRjVDt/xpOYjfp1jZ370SZhGN+kPV7VdyY6xqjvBOIuyDxxYUzpeF6AZhLzsE66sW2CuiukPpDI4fntp6WxAyh1IE6l0e8jFq5JwJt8zC1yazcNtr0EUk5QbMqa2PUSpY9Il6dJlm/QFno9jFl3HLFjcZOAGQY852F20yWVpktLfee71tOzFAzs6F94yqrYuMZnkp0eM533fEn4ZhVaNH/4/yWy+DSiWAAA='}, 'wamelink2014_s2.xlsx': {'sha256': 'caedacc76e245664d3acae0ecf92c7485712fb1d99292a3879a14bedf235a57f', 'gzip_b64': '...'}, 'Crop_recommendation.csv': {'sha256': '54a5a6e5408668e668667efc50de2fc867c1b875e0431b4f54dd331b0a109a4e', 'gzip_b64': '...'}, 'ds01_2020je006536rr.csv': {'sha256': '6a6e4359a543b315783bc672b9a173e4930592ba62c071d5c93b9a7b85584f8c', 'gzip_b64': '...'}}
for name,entry in SNAPSHOTS.items():
    b=gzip.decompress(base64.b64decode(entry['gzip_b64']))
    assert hashlib.sha256(b).hexdigest()==entry['sha256'], name
    (OUT/name).write_bytes(b)
DATA=pd.read_csv(OUT/'wamelink2014_pots.csv')
DATA.columns=DATA.columns.str.strip()
assert len(DATA)==840
assert DATA.groupby(['Species','Soil']).size().eq(20).all()
print('Verified 840 experimental pots; original data and ancillary references extracted.')


Verified 840 experimental pots; original data and ancillary references extracted.


## Source design and model
Wamelink et al. (2014), PLOS ONE, Table S2, https://doi.org/10.1371/journal.pone.0103138 (CC BY). Four food-crop candidates are available in this study: tomato, carrot, garden cress and rye. There are other, non-food species in the source; no conclusion is drawn for untested crops.

Each species/substrate has 20 pots in 20 blocks, with five seeds per pot. Pots, not individual seeds, define binary trials. With s successes and n pots, the chosen Beta(1,1) prior yields Beta(1+s,1+n−s). Report the mean and equal-tailed 95% credible interval. A Beta(0.5,0.5) sensitivity check is included.

Exchangeable independent pots are assumed; residual block/shared-greenhouse dependence and study heterogeneity are not modeled. This is one study, not independent external validation. Mean-based ordering is descriptive, not proof that one crop is statistically superior.


In [2]:
DOI='https://doi.org/10.1371/journal.pone.0103138'
SPECIES={'S. lycopersicum':'Tomato','D. carota':'Carrot','L. sativum':'Garden cress','S. cereale':'Rye'}
ENDPOINTS={'alive50':('nAlive','Alive at day 50'), 'germinated':('nGerminated','Germinated by day 50'), 'flowered':('nFlowers','Flower/bud formation by day 50')}

def posterior(k,n,prior=1.):
    a=prior+k;b=prior+n-k
    mean=a/(a+b);ci=beta.ppf([.025,.975],a,b)
    entropy=lambda p: -p*math.log(p)-(1-p)*math.log(1-p) if 0<p<1 else 0.
    expected_binary_entropy=float(digamma(a+b+1)-(a*digamma(a+1)+b*digamma(b+1))/(a+b))
    ig=(entropy(mean)-expected_binary_entropy)/math.log(2)
    return dict(successes=int(k),pots=int(n),observed_rate=k/n,alpha=a,beta=b,
                posterior_mean=mean,credible95=[float(ci[0]),float(ci[1])],
                expected_information_next_pot_bits=max(0.,ig))

def estimate(species='S. lycopersicum',endpoint='alive50',minimum=3,transfer=None):
    if species not in SPECIES:raise ValueError('Select a studied food crop')
    if endpoint not in ENDPOINTS:raise ValueError('Unknown endpoint')
    if isinstance(minimum,bool) or int(minimum)!=minimum or not 1<=int(minimum)<=5:raise ValueError('Minimum plants must be an integer from 1 to 5')
    minimum=int(minimum)
    if transfer is not None and (not math.isfinite(float(transfer)) or not 0<=float(transfer)<=1):raise ValueError('Transfer factor must be between 0 and 1')
    column,label=ENDPOINTS[endpoint];results={}
    for soil in ['Mars','Earth','Moon']:
        subset=DATA[(DATA.Species==species)&(DATA.Soil==soil)]
        if len(subset)!=20 or subset.Block.nunique()!=20:raise ValueError('Unexpected experimental design')
        vals=subset[column]
        if vals.isna().any() or not vals.between(0,5).all():raise ValueError('Invalid outcome values')
        k=int(vals.ge(minimum).sum());p=posterior(k,len(subset))
        p['jeffreys_prior_sensitivity']=posterior(k,len(subset),.5)['posterior_mean']
        p['seed_counts_descriptive_only']=int(vals.sum());results[soil]=p
    source=results['Mars'];projection=None
    if transfer is not None:
        q=float(transfer)
        projection=dict(status='ASSUMPTION-BASED, NOT VALIDATED ON MARS',transfer_factor=q,
            assumed_mean=q*source['posterior_mean'],
            conditional_credible95=[q*v for v in source['credible95']],
            formula='p_target = q * p_simulant',
            meaning='q is an assumed probability of avoiding additional mission-related failure given simulant success; the source data do not estimate q.',
            caveat='Interval conditions on fixed q and excludes uncertainty in transport, gravity, radiation, pressure, actual regolith and mission operations. This assumes additional stressors can only reduce source success.')
    return dict(species=species,crop=SPECIES[species],endpoint=endpoint,criterion=f'At least {minimum} of 5 plants per pot: {label.lower()}',
        horizon_days=50,minimum=minimum,results=results,mars_projection=projection,
        source_pots=[dict(block=int(row.Block),pot=int(row.Number),count=int(row[column]),success=bool(row[column]>=minimum)) for _,row in DATA[(DATA.Species==species)&(DATA.Soil=='Mars')].sort_values('Block').iterrows()],
        actual_mars_probability='NOT IDENTIFIED FROM AVAILABLE DATA',
        target_scope='A pot under the source greenhouse protocol with the selected simulant, seed material and success definition; not a whole harvest or mission.',
        prior='Beta(1,1), chosen uniform prior; sensitivity to Beta(0.5,0.5) reported',
        method='Binomial likelihood on 20 pot-level binary outcomes; Beta posterior. 95% equal-tailed credible interval.',
        study=dict(doi=DOI,supplement='Table S2',year=2014,conditions='Earth greenhouse; 50 days; five seeds per pot; study watering protocol; no added nutrient treatment',
            simulant='JSC-1A Mars regolith simulant as identified by the study',
            design='One study; 20 blocks, one pot per species/substrate in each block'),
        limitations=['Not validated on Mars or across independent studies','Exchangeable independent-pot model; residual block/greenhouse dependence is not modeled',
                    'No edible-yield, nutrition, toxicity, or food-safety endpoint','Simulant composition differs from actual regolith',
                    'No calibrated response surface for changing temperature, pH, irrigation or nutrients'])

def ranking(endpoint='alive50',minimum=3):
    out=[]
    for species in SPECIES:
        d=estimate(species,endpoint,minimum)
        out.append(dict(species=species,crop=d['crop'],**d['results']['Mars']))
    return sorted(out,key=lambda r:r['posterior_mean'],reverse=True)


In [3]:
ENDPOINT='alive50'  # alive50, germinated, flowered
MINIMUM=3  # At least this many plants out of 5 per pot; editable, 1 to 5
SPECIES_QUERY='S. lycopersicum'
comparison=pd.DataFrame(ranking(ENDPOINT,MINIMUM))
comparison['posterior_percent']=100*comparison.posterior_mean
comparison['credible95_low_percent']=comparison.credible95.map(lambda x:100*x[0])
comparison['credible95_high_percent']=comparison.credible95.map(lambda x:100*x[1])
comparison.to_csv(OUT/'crop_success_comparison.csv',index=False)
print(comparison[['crop','successes','pots','posterior_percent','credible95_low_percent','credible95_high_percent']].to_string(index=False))
selected=estimate(SPECIES_QUERY,ENDPOINT,MINIMUM)
(OUT/'selected_crop_estimate.json').write_text(json.dumps(selected,indent=2))
print('Criterion:',selected['criterion'])
print('Actual Mars:',selected['actual_mars_probability'])


        crop  successes  pots  posterior_percent  credible95_low_percent  credible95_high_percent
Garden cress         20    20          95.454545               83.890238                99.879512
         Rye         20    20          95.454545               83.890238                99.879512
      Carrot         19    20          90.909091               76.184009                98.825068
      Tomato         18    20          86.363636               69.622559                96.951103
Criterion: At least 3 of 5 plants per pot: alive at day 50
Actual Mars: NOT IDENTIFIED FROM AVAILABLE DATA


## Mission-transfer sensitivity, not a measured Mars probability
Assume p_target = q × p_simulant. Here q is an assumed probability of avoiding additional mission failure conditional on source-protocol success. It is not estimated by the experiment. The model assumes added stressors only reduce success and cannot represent benefits from improved engineering.

No default numerical Mars estimate is produced. Enable explicitly and choose q to explore assumptions. Conditional credible intervals treat q as fixed and omit uncertainty about gravity, radiation, pressure, actual regolith, water systems and mission operations.


In [4]:
ENABLE_TRANSFER=False
ASSUMED_Q=0.70  # Arbitrary sensitivity input, NOT a measured calibration
if ENABLE_TRANSFER:
    projection=estimate(SPECIES_QUERY,ENDPOINT,MINIMUM,ASSUMED_Q)['mars_projection']
    print(json.dumps(projection,indent=2))
    (OUT/'assumption_based_projection.json').write_text(json.dumps(projection,indent=2))
else:
    print('Mars projection disabled. Actual Mars probability is not identified.')


Mars projection disabled. Actual Mars probability is not identified.


## Expected information from another comparable pot
I(θ;Y) = H(Y) − Eθ[H(Y|θ)], in bits, under the beta posterior. It concerns repeating the same experimental protocol. It does not predict the information value of measuring perchlorates, gravity or changing the substrate. Decision alternatives, utility and cost are not included, so this is not full decision-theoretic VOI.

## Ancillary datasets
APXS bulk oxide measurements characterize real Mars material; Kaggle crop labels provide terrestrial context. Neither supplies cultivation-success outcomes. They are not used in the beta-binomial likelihood or converted to plant-available fertilizer values.


In [5]:
context=pd.read_csv(OUT/'ds01_2020je006536rr.csv',encoding='latin1')
soil=context[context.Strat_group.eq('Soil')]
soil.to_csv(OUT/'apxs_soil_context.csv',index=False)
crop=pd.read_csv(OUT/'Crop_recommendation.csv')
print('APXS analyses:',len(context),'Soil-classified:',len(soil))
print('Ancillary Crop rows:',len(crop),'NOT used as cultivation-success outcomes')
manifest=dict(question='Which plants could succeed on Mars?',
    experimental_source='https://doi.org/10.1371/journal.pone.0103138',
    apxs_source='https://doi.org/10.17189/1519050',
    crop_source='https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset',
    input_sha256={k:v['sha256'] for k,v in SNAPSHOTS.items()},
    endpoint=ENDPOINT,minimum_per_pot=MINIMUM,prior='Beta(1,1)',
    unit='pot',target_mars_probability='not identified',transfer_enabled=ENABLE_TRANSFER)
(OUT/'provenance.json').write_text(json.dumps(manifest,indent=2))


APXS analyses: 712 Soil-classified: 13
Ancillary Crop rows: 2200 NOT used as cultivation-success outcomes


## Evidence needed to answer the full Mars question
Add independent simulant studies and harvest endpoints; document treatment, cultivar, seed lot, perclorate/salt exposure, lighting, pressure, irrigation and substrate. Validate predictions by leaving entire experiments out. A Mars transfer model needs data or justified prior uncertainty for additional mission effects. Do not multiply arbitrary risk factors and call the result empirical.

The earlier crop-label interval demonstration is superseded as the primary research model. It did not answer the cultivation-success question.


## Joint temperature and soil-pH interval analysis
This independent LP module combines greenhouse temperature, substrate pH and air humidity for the 22 Crop labels. Marginal and selected pairwise overlap frequencies constrain eight worlds. Greedy selection minimizes interval width; endpoint-witness pruning is checked against full reoptimization. Windows are analysis settings, not validated physiological limits.
The modeled event is joint overlap of a terrestrial dataset row with the scenario. It is NOT germination or Mars success. Those probabilities remain unidentified [0,1] for these Crop labels without response/transfer evidence. NPK availability, salinity and perchlorates remain unlinked rather than imputed. The direct joint frequency is an audit of the LP bounds; intervals are not sampling confidence intervals.


In [6]:
"""Independent LP bounds and endpoint-witness pruning for joint Crop overlap.
The event is measured environmental overlap, never biological cultivation success.
"""
from itertools import product,combinations
from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.optimize import linprog
CROP=pd.read_csv(OUT/'Crop_recommendation.csv')
FEATURES=['temperature','ph','humidity']
WORLD=np.array(list(product([0.,1.],repeat=3)))
OBJECTIVE=np.prod(WORLD,axis=1)

def solve_bounds(observations,budget=2):
    base_A=[np.ones(8)]+[WORLD[:,i] for i in range(3)]
    base_b=[1.]+observations.mean(axis=0).tolist()
    candidates=[]
    for i,j in combinations(range(3),2):
        candidates.append((FEATURES[i]+' & '+FEATURES[j],WORLD[:,i]*WORLD[:,j],float((observations[:,i]*observations[:,j]).mean())))
    def run(prune):
        A=list(base_A);b=list(base_b);pool=list(candidates);calls=0;skips=0
        def lp(a,bb,sign):
            nonlocal calls
            calls+=1;r=linprog(sign*OBJECTIVE,A_eq=np.array(a),b_eq=np.array(bb),bounds=(0,None),method='highs')
            if not r.success:raise ValueError('Infeasible empirical constraints')
            return float(OBJECTIVE@r.x),r.x
        lower,xl=lp(A,b,1);upper,xu=lp(A,b,-1)
        history=[dict(step=0,lower=max(0.,lower),upper=min(1.,upper),width=upper-lower,selected='marginals')]
        for step in range(1,budget+1):
            trial=[]
            for name,row,value in pool:
                if prune and abs(row@xl-value)<=1e-8:l,xxl=lower,xl;skips+=1
                else:l,xxl=lp(A+[row],b+[value],1)
                if prune and abs(row@xu-value)<=1e-8:u,xxu=upper,xu;skips+=1
                else:u,xxu=lp(A+[row],b+[value],-1)
                trial.append((u-l,name,row,value,l,u,xxl,xxu))
            _,name,row,value,lower,upper,xl,xu=min(trial,key=lambda t:(round(t[0],10),t[1]))
            A.append(row);b.append(value);pool=[c for c in pool if c[0]!=name]
            history.append(dict(step=step,lower=max(0.,lower),upper=min(1.,upper),width=upper-lower,selected=name))
        return dict(history=history,solver_calls=calls,pruned_endpoint_solves=skips)
    full=run(False);pruned=run(True)
    agree=all(abs(a[k]-b[k])<1e-7 for a,b in zip(full['history'],pruned['history']) for k in ['lower','upper'])
    joint=float(np.prod(observations,axis=1).mean())
    for h in pruned['history']:
        if not h['lower']-1e-7<=joint<=h['upper']+1e-7:raise ValueError('Empirical joint outside LP bounds')
    return full,pruned,agree,joint

def scenario(temperature=21.1,ph=7.7,humidity=65.,temperature_tolerance=5.,ph_tolerance=.5,humidity_tolerance=15.,budget=2):
    inputs=[temperature,ph,humidity,temperature_tolerance,ph_tolerance,humidity_tolerance]
    try:inputs=list(map(float,inputs))
    except (ValueError,TypeError):raise ValueError('Scenario inputs must be numeric')
    if not all(math.isfinite(v) for v in inputs):raise ValueError('Scenario must contain finite values')
    t,p,h,tt,pt,ht=inputs
    if not -100<=t<=100 or not 0<=p<=14 or not 0<=h<=100:raise ValueError('Scenario value outside allowed range')
    if not 0<tt<=50 or not 0<pt<=14 or not 0<ht<=100:raise ValueError('Window widths must be positive and within limits')
    if isinstance(budget,bool) or not isinstance(budget,int) or not 0<=budget<=3:raise ValueError('Budget must be an integer from 0 to 3')
    center=np.array([t,p,h]);tol=np.array([tt,pt,ht]);rows=[]
    for label,g in CROP.groupby('label'):
        obs=(np.abs(g[FEATURES].to_numpy()-center)<=tol).astype(float)
        full,pruned,agree,joint=solve_bounds(obs,budget)
        last=pruned['history'][-1]
        outside=[f for f in FEATURES if center[FEATURES.index(f)]<g[f].min() or center[FEATURES.index(f)]>g[f].max()]
        rows.append(dict(crop=label,lower=last['lower'],upper=last['upper'],width=last['width'],
            empirical_joint=joint,matching_rows=int(np.prod(obs,axis=1).sum()),dataset_rows=len(g),
            marginal_overlap=dict(zip(FEATURES,obs.mean(axis=0).tolist())),
            full=full,pruned=pruned,equivalent_bounds=agree,
            solver_calls_saved=full['solver_calls']-pruned['solver_calls'],
            center_outside_observed_ranges=outside,
            actual_mars_success_bounds=[0.,1.],
            evidence_link_to_cultivation_success='UNAVAILABLE: no matching experimental outcome labels'))
    rows.sort(key=lambda r:(-r['lower'],-r['upper'],r['crop']))
    return dict(rows=rows,scenario=dict(zip(FEATURES,center.tolist())),windows=dict(zip(FEATURES,tol.tolist())),budget=budget,
        modeled_event='A randomly selected terrestrial Crop row for this crop lies inside all three scenario windows.',
        interpretation='Partial-identification bounds from empirical marginals and selected pairwise constraints; not success probabilities or biological tolerance limits.',
        all_bounds_agree=all(r['equivalent_bounds'] for r in rows),
        solver_calls_full=sum(r['full']['solver_calls'] for r in rows),
        solver_calls_pruned=sum(r['pruned']['solver_calls'] for r in rows),
        soil_evidence=dict(ph='Phoenix 7.7 +/-0.5 is an optional site-specific reference; scenario pH is editable',
            NPK='Not integrated: plant-available fractions and analytical units are not harmonized',
            salinity='Not integrated: no matched Crop electrical-conductivity feature',
            APXS='Real bulk chemistry shown as context; no conversion to available fertilizer'),
        mars_cultivation_success='NOT IDENTIFIED: [0,1] without additional response/transfer assumptions',
        method='Finite-world LP (8 worlds), greedy pairwise-constraint selection by immediate width reduction, endpoint-witness pruning; independently implemented, not verified as an exact reproduction of earlier work.')


In [7]:
joint=scenario(temperature=21.1,ph=7.7,humidity=65,temperature_tolerance=5,ph_tolerance=.5,humidity_tolerance=15,budget=2)
assert joint["all_bounds_agree"]
summary=pd.DataFrame([{k:v for k,v in r.items() if k in ["crop","lower","upper","matching_rows","dataset_rows","solver_calls_saved"]} for r in joint["rows"]])
summary.to_csv(OUT/"joint_scenario_lp_bounds.csv",index=False)
print(summary.to_string(index=False))
print("Solver calls:",joint["solver_calls_full"],"full;",joint["solver_calls_pruned"],"pruned")
print("Actual Mars cultivation success:",joint["mars_cultivation_success"])


       crop  lower  upper  matching_rows  dataset_rows  solver_calls_saved
     cotton   0.24   0.24             24           100                   8
     lentil   0.20   0.20             20           100                   8
     coffee   0.12   0.12             12           100                   8
       jute   0.06   0.12              8           100                   0
  blackgram   0.04   0.04              4           100                   8
  mothbeans   0.00   0.05              3           100                   0
 pigeonpeas   0.00   0.01              0           100                   0
      apple   0.00   0.00              0           100                  10
     banana   0.00   0.00              0           100                   6
   chickpea   0.00   0.00              0           100                  10
    coconut   0.00   0.00              0           100                  10
     grapes   0.00   0.00              0           100                  10
kidneybeans   0.00   0.00

## Mathematical foundation: interval queries and endpoint pruning
For a fixed crop $c$, define window events $T$ (temperature), $H$ (soil pH), and $R$ (air relative humidity). $Q=T\cap H\cap R$ is dataset overlap, not cultivation success.

There are eight worlds $w\in\{0,1\}^3$, with masses $\pi_w\ge0$, $\sum_w\pi_w=1$. For each selected event $E$:
$$\sum_w \mathbf{1}_E(w)\pi_w=\widehat p_c(E),\qquad \widehat p_c(E)=\frac{1}{n_c}\sum_j\mathbf{1}_E(x_{cj}).$$
The initial constraints are marginal frequencies; candidates are the three pairwise conjunctions. With feasible set $F_k$ and $q_w=\mathbf{1}_Q(w)$:
$$L_k=\min_{\pi\in F_k}q^T\pi,\quad U_k=\max_{\pi\in F_k}q^T\pi,\quad W_k=U_k-L_k.$$
Adding a consistent constraint shrinks $F_k$, so $L_{k+1}\ge L_k$ and $U_{k+1}\le U_k$. A baseline analytic check is:
$$\max(0,\widehat p(T)+\widehat p(H)+\widehat p(R)-2)\le P(Q\mid c)\le\min(\widehat p(T),\widehat p(H),\widehat p(R)).$$
Greedy selection uses $\Delta_k(a)=W_k-W(F_k\cap a)$, choosing maximum immediate width reduction. It is neither a global optimization guarantee nor full VOI.

**Endpoint-pruning argument:** if an old minimizing optimizer remains feasible after adding a constraint, its objective value is still attainable. The reduced feasible set cannot have a smaller minimum. Thus the minimum is unchanged. The maximum follows by the same argument with reversed inequalities. If the stored witness violates the constraint, re-solve. Alternative old optimizers are not searched, so pruning opportunities may be missed. Exact mathematics is qualified numerically: witness residual tolerance $10^{-8}$, paired-bound comparison tolerance $10^{-7}$, stable ties after rounding widths to ten decimals.

The direct triple frequency is an audit, not a selected constraint; it must remain in the computed bounds. Plug-in identification bounds are not sampling confidence intervals.

**Outcome model:** for 20 binary pot trials, $s\mid\theta\sim\mathrm{Binomial}(n,\theta)$, $\theta\sim\mathrm{Beta}(\alpha,\beta)$, hence $\theta\mid s\sim\mathrm{Beta}(\alpha+s,\beta+n-s)$, with chosen $\alpha=\beta=1$. The 95% interval uses posterior quantiles. This is success under the simulant protocol, not actual Mars. Residual block/shared-greenhouse dependence is not modeled.

Expected information from another comparable pot is $I(\theta;Y)=H(Y)-E_\theta[H(Y\mid\theta)]$, in bits. This is distinct from LP width reduction.

**Identifiability:** without response/transfer assumptions, $P(S_{Mars})\in[0,1]$. The optional $p_{target}=q\,p_{simulant}$ is sensitivity analysis with fixed, assumed $q$, not an empirical Mars estimate.

This is an independent implementation of the methodological family of interval queries, greedy constraint selection and endpoint pruning. Exact equivalence to the unavailable previous code has not been verified. Numerical method: [SciPy HiGHS linear programming](https://docs.scipy.org/doc/scipy/reference/optimize.linprog-highs.html).
